In [3]:
# !pip install playwright

In [4]:
# !playwright install

# !playwright install chromium

In [ ]:
from playwright.async_api import async_playwright

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()

await page.goto("https://www.whitehouse.gov/presidential-actions/executive-orders/")

await page.get_by_role("link", name="Next", exact=True).click()

print(page.url)

posts = page.locator("li.wp-block-post")

rows = []

for post in await posts.all():
    title = await post.locator("h2.wp-block-post-title a").inner_text()
    link = await post.locator("h2.wp-block-post-title a").get_attribute("href")
    date = await post.locator(".wp-block-post-date").inner_text()
    rows.append({"title": title, "date": date, "link": link})

import pandas as pd
df = pd.DataFrame(rows)
df

https://www.whitehouse.gov/presidential-actions/executive-orders/page/2/


,title,date,link
0,Supporting America’s Ranchers,"September 4, 2026",https://www.whitehouse.gov/presidential-action...
1,Promoting Fair Competition In Livestock Market...,"September 4, 2026",https://www.whitehouse.gov/presidential-action...
2,Honoring the American History of the Great Lak...,"August 27, 2026",https://www.whitehouse.gov/presidential-action...
3,Declaring a National Emergency to Secure the U...,"August 26, 2026",https://www.whitehouse.gov/presidential-action...
4,Delivering Gold Standard Childhood Vaccine Rec...,"August 10, 2026",https://www.whitehouse.gov/presidential-action...
5,Continuing to Protect the Meaning and Value of...,"August 6, 2026",https://www.whitehouse.gov/presidential-action...
6,Ending Birth Tourism,"August 6, 2026",https://www.whitehouse.gov/presidential-action...
7,Establishing the President’s Military Spouse C...,"August 3, 2026",https://www.whitehouse.gov/presidential-action...
8,Restoring Trust in the Smithsonian Institution,"July 24, 2026",https://www.whitehouse.gov/presidential-action...
9,Securing America’s Defense Supply Chains and E...,"July 20, 2026",https://www.whitehouse.gov/presidential-action...


In [37]:
async def scrape_page(page):
    rows = []
    posts = page.locator("li.wp-block-post")

    for post in await posts.all():
        link_element = post.locator("h2.wp-block-post-title a")

        name = await link_element.inner_text()
        link = await link_element.get_attribute("href")
        date = await post.locator(".wp-block-post-date").inner_text()

        rows.append({"name": name, "date": date, "link": link})

    df=pd.DataFrame(rows)

    return df

In [38]:
p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()

await page.goto("https://www.whitehouse.gov/presidential-actions/executive-orders/")

first_page = await scrape_page(page)
first_page

,name,date,link
0,Emergency Tax Relief on Diesel Fuel,"October 5, 2026",https://www.whitehouse.gov/presidential-action...
1,Eliminating Disease-Carrying Pests And Restori...,"September 29, 2026",https://www.whitehouse.gov/presidential-action...
2,Inaugurating The Era Of Super Intelligence,"September 29, 2026",https://www.whitehouse.gov/presidential-action...
3,Streamlining Access to Government Services Thr...,"September 29, 2026",https://www.whitehouse.gov/presidential-action...
4,Enhancing Program Integrity and Integrity and ...,"September 18, 2026",https://www.whitehouse.gov/presidential-action...
5,RESTORING AMERICAN SALTWATER ANGLING AND RECRE...,"September 17, 2026",https://www.whitehouse.gov/presidential-action...
6,Reinvigorating America’s Hunting Heritage,"September 17, 2026",https://www.whitehouse.gov/presidential-action...
7,Providing Meaningful Water Quality Improvement...,"September 16, 2026",https://www.whitehouse.gov/presidential-action...
8,Accelerating Access To Veterans’ Benefits And ...,"September 8, 2026",https://www.whitehouse.gov/presidential-action...
9,Adjusting Certain Delegations Under the Defens...,"September 8, 2026",https://www.whitehouse.gov/presidential-action...


In [41]:
import asyncio
import random

p = await async_playwright().start()
browser = await p.chromium.launch(headless=False, slow_mo=500)
page = await browser.new_page()

await page.goto("https://www.whitehouse.gov/presidential-actions/executive-orders/")

dfs = []
page_num = 1

while True:
    df = await scrape_page(page)
    dfs.append(df)

    # Stop if there's no "Next" link
    next_link = page.get_by_role("link", name="Next", exact=True)
    if await next_link.count() == 0:
        print("Reached the last page.")
        break
        
    # Wait, click "Next", and wait for the new page to load
    await asyncio.sleep(random.uniform(1, 3))
    await next_link.click()
    page_num += 1
    await page.wait_for_url(f"**/page/{page_num}/")


master_df=pd.concat(dfs, ignore_index=True)

master_df.shape

Reached the last page.


(199, 3)